<a href="https://colab.research.google.com/github/Jyotsna135-bit/Generative-AI-/blob/main/vllm_documented.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Deployment using vLLM (Google Gemma 2)

This notebook deploys a large language model using vLLM — a different inference engine compared to Ollama.

**Model used:** `google/gemma-2-2b-it` — same Gemma 2 model as the Ollama notebook, but loaded here through vLLM directly from HuggingFace.

**What's vLLM?** It's an open-source inference engine built for speed and efficiency. The main thing that makes it faster than standard inference is something called **PagedAttention** — a smarter way of managing GPU memory that lets it handle more requests at the same time.

The key point: both Ollama and vLLM expose the same OpenAI-compatible API, so the Python code to query them is almost identical.

> Make sure the runtime is set to **T4 GPU** before running — vLLM requires a GPU.

## Checking the GPU

Quick sanity check to confirm a GPU is available before we do anything else. If this shows no GPU, go to Runtime → Change runtime type → T4 GPU.

In [ ]:
!nvidia-smi

## Installing vLLM

vLLM is a Python package so we just pip install it. It's a larger install than Ollama and takes a few minutes — it pulls in PyTorch and a bunch of other dependencies.

In [ ]:
!pip install vllm -q

## Starting the vLLM Server

vLLM has a built-in API server that we start as a background process. It automatically downloads the Gemma 2 model from HuggingFace on first run, then loads it into GPU memory.

The server runs on port `8000` and exposes the same `/v1/chat/completions` endpoint as OpenAI. We wait in a loop until it's ready — loading the model can take 2-3 minutes.

In [3]:
import subprocess, time, requests

process = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", "google/gemma-2-2b-it",
        "--dtype", "float16",
        "--max-model-len", "2048",
        "--port", "8000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print("waiting for vLLM to load the model, this takes a couple minutes...")
for _ in range(180):
    try:
        r = requests.get("http://localhost:8000/health")
        if r.status_code == 200:
            print("vLLM is running")
            break
    except:
        time.sleep(1)
else:
    print("timed out — check if GPU is enabled")

waiting for vLLM to load the model, this takes a couple minutes...
timed out — check if GPU is enabled


## curl Client

Same curl request format as Ollama — just a different port (`8000` instead of `11434`) and the full HuggingFace model name instead of the short Ollama name. The API structure is identical.

In [4]:
%%bash
curl -s http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer token" \
  -d '{
    "model": "google/gemma-2-2b-it",
    "messages": [
      {"role": "user", "content": "What is a large language model? Keep it short."}
    ],
    "max_tokens": 200
  }' | python3 -c "import json,sys; print(json.load(sys.stdin)['choices'][0]['message']['content'])"

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/usr/lib/python3.12/json/__init__.py", line 293, in load
    return loads(fp.read(),
           ^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/json/__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/json/decoder.py", line 338, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/json/decoder.py", line 356, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecting value: line 1 column 1 (char 0)


CalledProcessError: Command 'b'curl -s http://localhost:8000/v1/chat/completions \\\n  -H "Content-Type: application/json" \\\n  -H "Authorization: Bearer token" \\\n  -d \'{\n    "model": "google/gemma-2-2b-it",\n    "messages": [\n      {"role": "user", "content": "What is a large language model? Keep it short."}\n    ],\n    "max_tokens": 200\n  }\' | python3 -c "import json,sys; print(json.load(sys.stdin)[\'choices\'][0][\'message\'][\'content\'])"\n'' returned non-zero exit status 1.

## Python Client (OpenAI SDK)

Same `openai` library as before, just pointed at port `8000`. This is the whole point of the OpenAI-compatible API — you can switch between Ollama, vLLM, or the actual OpenAI service by changing just the `base_url` and `model` name. Nothing else in your code needs to change.

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="token",
)

response = client.chat.completions.create(
    model="google/gemma-2-2b-it",
    messages=[
        {"role": "user", "content": "What is a large language model? Keep it short."}
    ],
    max_tokens=200
)

print(response.choices[0].message.content)

## Streaming

vLLM supports streaming out of the box. Same `stream=True` flag, same chunk-by-chunk printing — works exactly like it did with Ollama.

In [ ]:
stream = client.chat.completions.create(
    model="google/gemma-2-2b-it",
    messages=[
        {"role": "user", "content": "Explain how transformer architecture works in simple terms."}
    ],
    max_tokens=300,
    stream=True
)

for chunk in stream:
    text = chunk.choices[0].delta.content
    if text:
        print(text, end="", flush=True)

## Multi-turn Conversation

Same approach as the Ollama notebooks — we keep a running list of messages and pass the full history on every request. The model itself is stateless, so we're the ones responsible for maintaining context between turns.

In [ ]:
messages = [{"role": "system", "content": "You are a helpful assistant."}]

def chat(user_input):
    messages.append({"role": "user", "content": user_input})
    res = client.chat.completions.create(
        model="google/gemma-2-2b-it",
        messages=messages,
        max_tokens=300
    )
    reply = res.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})
    return reply

print(chat("What is vLLM and why is it faster than regular inference?"))
print(chat("What is PagedAttention?"))